In [ ]:
import os
from IPython.display import clear_output

# 1. Cài đặt Unsloth cơ bản
!pip install unsloth unsloth-zoo

# 2. Cài đặt bổ sung các thư viện lõi
!pip install msgspec tyro cut_cross_entropy torchao==0.13.0

# 3. Cài đặt các thư viện ML với phiên bản chuẩn
!pip install "trl>=0.18.2,<=0.24.0" "datasets<4.4.0,>=3.4.1" transformers==4.56.2 modelscope

# 4. GIẢI QUYẾT TRIỆT ĐỂ LỖI ĐỎ CỦA KAGGLE
# - Cài thêm thư viện thiếu cho bigframes
!pip install google-cloud-bigquery-storage<3.0.0,>=2.30.0
# - Ép đồng bộ bộ 3 thư viện quản lý file về cùng bản 2025.9.0
!pip install fsspec==2025.9.0 gcsfs==2025.9.0 s3fs==2025.9.0


In [ ]:
from unsloth import FastLanguageModel 
import torch
from datasets import load_dataset
from unsloth.chat_templates import get_chat_template
from trl import SFTTrainer
from transformers import TrainingArguments
from transformers import TextStreamer
import gc
max_seq_length = 1024
dtype = None
# ĐÃ FIX: BẬT load_in_4bit = True
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen3-4B-Instruct-2507-unsloth-bnb-4bit",
    dtype = dtype,
    max_seq_length = max_seq_length,
    load_in_4bit = True,
    full_finetuning = False,
)

# Khối 2: Gắn LoRA (Giữ nguyên của bạn)
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    random_state = 3407,
)

In [ ]:
import json

def calculate_detailed_metrics(y_true_list, y_pred_list, generation_times, generated_tokens_count):
    total_samples = len(y_true_list)
    if total_samples == 0:
        return {}

    valid_json_count = 0
    schema_conform_count = 0
    exact_dict_matches = 0
    
    # Tracking cho Task-Oriented Dialogue
    joint_goals_achieved = 0
    correct_intents = 0
    
    # Tracking cho Top-level (Tên, SĐT, Intent, Note...)
    TP_top, FP_top, FN_top = 0, 0, 0
    
    # Tracking riêng cho mảng Items (Sản phẩm, số lượng)
    TP_item, FP_item, FN_item = 0, 0, 0

    expected_keys = {"intent", "customer_name", "phone_number", "location_tag", "payment_intent", "delivery_note", "order_note", "items", "inquiries"}

    for true_str, pred_str in zip(y_true_list, y_pred_list):
        try:
            true_json = json.loads(true_str)
        except json.JSONDecodeError:
            continue # Ground truth lỗi thì bỏ qua
            
        try:
            pred_json = json.loads(pred_str)
            valid_json_count += 1
        except json.JSONDecodeError:
            # Lỗi parse JSON -> Phạt nặng, ghi nhận toàn bộ là False Negative/False Positive
            FP_top += len(expected_keys)
            continue

        # 1. Schema Conformance
        if set(pred_json.keys()) == expected_keys:
            schema_conform_count += 1

        # 2. Exact Match (So sánh bằng Dict)
        if true_json == pred_json:
            exact_dict_matches += 1

        # 3. Intent Accuracy
        if true_json.get("intent") == pred_json.get("intent"):
            correct_intents += 1

        # Cờ đánh dấu JGA cho từng sample
        is_jga_correct = True

        # 4. Top-Level Slot Metrics
        for key in expected_keys:
            if key in ["items", "inquiries"]: 
                continue # Xử lý list riêng
                
            val_true = true_json.get(key)
            val_pred = pred_json.get(key)
            
            if val_true == val_pred and val_true is not None:
                TP_top += 1
            elif val_pred is not None and val_true != val_pred:
                FP_top += 1
                is_jga_correct = False # Sai 1 slot là hỏng Joint Goal
            elif val_true is not None and val_pred in [None, ""]:
                FN_top += 1
                is_jga_correct = False

        # 5. Deep List Evaluation (Items)
        true_items = true_json.get("items", [])
        pred_items = pred_json.get("items", [])
        
        def hashable_items(item_list):
            if not isinstance(item_list, list): return []
            res = []
            for it in item_list:
                if isinstance(it, dict):
                    # Loại bỏ các key có giá trị None để tập trung vào dữ liệu thực
                    res.append(tuple(sorted((str(k), str(v)) for k, v in it.items() if v is not None)))
            return res
        
        true_set = set(hashable_items(true_items))
        pred_set = set(hashable_items(pred_items))
        
        tp_i = len(true_set & pred_set)
        fp_i = len(pred_set - true_set)
        fn_i = len(true_set - pred_set)
        
        TP_item += tp_i
        FP_item += fp_i
        FN_item += fn_i
        
        if fp_i > 0 or fn_i > 0:
            is_jga_correct = False

        # Ghi nhận JGA nếu sample này vượt qua mọi bài test slot
        if is_jga_correct:
            joint_goals_achieved += 1

    # ==========================================
    # TÍNH TOÁN CÁC CHỈ SỐ RATE ĐẦU RA
    # ==========================================
    validity_rate = valid_json_count / total_samples if total_samples else 0
    schema_rate = schema_conform_count / total_samples if total_samples else 0
    em_rate = exact_dict_matches / total_samples if total_samples else 0
    jga_rate = joint_goals_achieved / total_samples if total_samples else 0
    intent_acc = correct_intents / total_samples if total_samples else 0
    
    # Hàm tính P, R, F1 và SER
    def get_detailed_metrics(tp, fp, fn):
        prec = tp / (tp + fp) if (tp + fp) > 0 else 0
        rec = tp / (tp + fn) if (tp + fn) > 0 else 0
        f1 = 2 * (prec * rec) / (prec + rec) if (prec + rec) > 0 else 0
        # Slot Error Rate = (FP + FN) / Total True Slots
        ser = (fp + fn) / (tp + fn) if (tp + fn) > 0 else 0 
        return prec, rec, f1, ser
        
    top_p, top_r, top_f1, top_ser = get_detailed_metrics(TP_top, FP_top, FN_top)
    item_p, item_r, item_f1, item_ser = get_detailed_metrics(TP_item, FP_item, FN_item)
    
    # Tính Tổng thể (Overall)
    ov_tp = TP_top + TP_item
    ov_fp = FP_top + FP_item
    ov_fn = FN_top + FN_item
    ov_p, ov_r, ov_f1, ov_ser = get_detailed_metrics(ov_tp, ov_fp, ov_fn)

    # Tính TPS (Tokens per Second)
    total_time = sum(generation_times)
    total_tokens = sum(generated_tokens_count)
    tps = total_tokens / total_time if total_time > 0 else 0

    return {
        "Total Test Samples": total_samples,
        "JSON Validity Rate": f"{validity_rate:.2%}",
        "Schema Conformance": f"{schema_rate:.2%}",
        "Intent Accuracy": f"{intent_acc:.2%}",
        "Joint Goal Acc (JGA)": f"{jga_rate:.2%}",
        "Dict Exact Match": f"{em_rate:.2%}",
        "-----------------------": "-----------------------",
        "[TOP-LEVEL] Precision": f"{top_p:.2%}",
        "[TOP-LEVEL] Recall": f"{top_r:.2%}",
        "[TOP-LEVEL] F1-Score": f"{top_f1:.2%}",
        "[TOP-LEVEL] Slot Error Rate": f"{top_ser:.2%}",
        "----------------------- ": "-----------------------",
        "[ITEM-LEVEL] Precision": f"{item_p:.2%}",
        "[ITEM-LEVEL] Recall": f"{item_r:.2%}",
        "[ITEM-LEVEL] F1-Score": f"{item_f1:.2%}",
        "[ITEM-LEVEL] Slot Error Rate": f"{item_ser:.2%}",
        "-----------------------  ": "-----------------------",
        "[OVERALL] F1-Score": f"{ov_f1:.2%}",
        "[OVERALL] Slot Error Rate": f"{ov_ser:.2%}",
        "Avg TPS (Tokens/s)": f"{tps:.2f}"
    }

In [ ]:
from transformers import EarlyStoppingCallback
from unsloth.chat_templates import get_chat_template, standardize_data_formats, train_on_responses_only
from trl import SFTTrainer
from unsloth import unsloth_train, is_bfloat16_supported
from transformers import TrainingArguments 
import torch
from datasets import load_dataset
import random
import shutil
# ==========================================
# 1. LOAD, SHUFFLE VÀ CHIA DATASET
# ==========================================
tokenizer = get_chat_template(tokenizer, chat_template="chatml")

# Load data và XÁO TRỘN (Shuffle) triệt để
dataset = load_dataset("json", data_files={"train": "/kaggle/input/datasets/xvmhieu/alo-315-yeye/trainning_ds_31-5.jsonl"}, split="train")
dataset = standardize_data_formats(dataset)
dataset = dataset.shuffle(seed=3407)

def formatting_prompts_func(examples):
    convos = examples["conversations"] if "conversations" in examples else examples["messages"]
    texts = [tokenizer.apply_chat_template(convo, tokenize=False, add_generation_prompt=False) for convo in convos]
    return { "text" : texts }

dataset = dataset.map(formatting_prompts_func, batched=True)
dataset_split = dataset.train_test_split(test_size=0.15, seed=3407)

train_dataset = dataset_split["train"]
eval_dataset = dataset_split["test"]

# ==========================================
# 2. CẤU HÌNH TRAINER VỚI EARLY STOPPING
# ==========================================
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train_dataset,  
    eval_dataset = eval_dataset,    
    dataset_text_field = "text",  
    max_seq_length = 1024,
    args = TrainingArguments(
        per_device_train_batch_size = 1,
        per_device_eval_batch_size = 1, 
        gradient_accumulation_steps = 16, 
        warmup_steps = 10,
        num_train_epochs = 15,       
        learning_rate = 2e-4,        
        
        # CẤU HÌNH ĐÁNH GIÁ & EARLY STOPPING
        eval_strategy = "steps",       
        eval_steps = 30,               
        save_strategy = "steps",       
        save_steps = 30,
        save_total_limit = 2,          
        load_best_model_at_end = True, 
        save_only_model = True,
        metric_for_best_model = "eval_loss",
        greater_is_better = False,     

        adam_epsilon = 1e-5,         
        optim = "paged_adamw_32bit", 
        max_grad_norm = 0.3,         
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 5,           
        weight_decay = 0.01,
        lr_scheduler_type = "cosine",
        seed = 3407,
        output_dir = "/kaggle/working/outputs", 
        report_to = "none", 
    ),
    callbacks=[EarlyStoppingCallback(early_stopping_patience=5)]
)

trainer = train_on_responses_only(
    trainer,
    instruction_part = "<|im_start|>user\n",
    response_part = "<|im_start|>assistant\n",
)

print("Bắt đầu huấn luyện với Phanh tự động (Early Stopping) và lỗi gradient accumulation(un_sloth train)...")
trainer_stats = unsloth_train(trainer)

best_model_path = "/kaggle/working/best_model_lora"
model.save_pretrained(best_model_path)
tokenizer.save_pretrained(best_model_path)
print(f"✅ Đã lưu thành công LoRA weights tốt nhất (dung lượng nhẹ) tại: {best_model_path}")

# Đường dẫn thư mục output
output_dir = "/kaggle/working/outputs"

# clean up output 
if os.path.exists(output_dir):
    for folder in os.listdir(output_dir):
        folder_path = os.path.join(output_dir, folder)
        if os.path.isdir(folder_path) and "checkpoint" in folder:
            print(f"🧹 Đang xóa checkpoint thừa: {folder}")
            shutil.rmtree(folder_path)

print("✅ Đã dọn dẹp sạch bộ nhớ working directory.")

In [ ]:
import json
import torch
from unsloth import FastLanguageModel
from tqdm.auto import tqdm # Thư viện này giúp hiện thanh tiến trình cho đẹp
import time

# ==========================================
# 2. CHUẨN BỊ MÔ HÌNH VÀ TẬP TEST
# ==========================================
print("🚀 BẬT CHẾ ĐỘ SUY LUẬN NHANH VÀ BẮT ĐẦU INFERENCE...")
FastLanguageModel.for_inference(model) # Tối ưu hóa bộ nhớ và tốc độ cho inference

y_true_list = []
y_pred_list = []
generation_times = []    
generated_tokens_count = []
# Tắt warning để console đỡ rối
import warnings
warnings.filterwarnings('ignore')

for item in tqdm(eval_dataset, desc="Đang chạy dự đoán (Inference)"):
    conversations = item["conversations"] if "conversations" in item else item["messages"]
    
    prompt_messages = []
    ground_truth_text = ""
    
    for msg in conversations:
        if msg["role"] in ["system", "user"]:
            prompt_messages.append(msg)
        elif msg["role"] == "assistant":
            ground_truth_text = msg["content"]
            
    y_true_list.append(ground_truth_text)
    
    inputs = tokenizer.apply_chat_template(
        prompt_messages,
        tokenize = True,
        add_generation_prompt = True,
        return_tensors = "pt"
    ).to("cuda")
    
    # Bắt đầu bấm giờ
    start_time = time.time()
    
    outputs = model.generate(
        input_ids = inputs,
        max_new_tokens = 1024,
        use_cache = True,
        do_sample = False,       
        temperature = 0.0,       
        pad_token_id = tokenizer.pad_token_id,
        eos_token_id = tokenizer.eos_token_id,
    )
    
    # Dừng bấm giờ
    end_time = time.time()
    
    prompt_length = inputs.shape[1]
    generated_tokens = outputs[0][prompt_length:]
    pred_text = tokenizer.decode(generated_tokens, skip_special_tokens=True)
    
    # Lưu lại metrics hiệu năng
    generation_times.append(end_time - start_time)
    generated_tokens_count.append(len(generated_tokens))
    
    y_pred_list.append(pred_text)




In [ ]:
# ==========================================
# 3. TÍNH TOÁN VÀ IN BÁO CÁO
# ==========================================
print("\n✅ ĐÃ INFERENCE XONG! ĐANG TÍNH TOÁN METRICS...")
metrics_report = calculate_detailed_metrics(y_true_list, y_pred_list, generation_times, generated_tokens)

print("\n" + "="*40)
print("📊 BẢNG KẾT QUẢ ĐÁNH GIÁ (EVALUATION REPORT)")
print("="*40)
for key, value in metrics_report.items():
    print(f" {key:<20}: {value}")
print("="*40)

# (Tùy chọn) Lưu file log để xem lại câu nào sai
with open("/kaggle/working/test_evaluation_log.jsonl", "w", encoding="utf-8") as f:
    for t, p in zip(y_true_list, y_pred_list):
        log_item = {"ground_truth": t, "prediction": p}
        f.write(json.dumps(log_item, ensure_ascii=False) + "\n")
print("Đã lưu chi tiết so sánh vào /kaggle/working/test_evaluation_log.jsonl")

In [ ]:
gc.collect()
torch.cuda.empty_cache()
# ==========================================
# PHẦN 5: XUẤT RA ĐỊNH DẠNG GGUF (CHO OLLAMA)
# ==========================================
REPO_NAME = "zxcvmh666/model_gAO_extract"
model.push_to_hub_gguf(
    REPO_NAME,
    tokenizer,
    quantization_method = "q4_k_m",
    token = HF_TOKEN,
)
print(f"✅ Đã xuất xong file GGUF! Bạn có thể vào link {REPO_NAME} để tải file về cho Ollama.")